# E3.2 · Governing autonomy rather than approving tools

**Function E — Governance, Risk, Compliance & the CISO Office → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

---

**Risk.** A per-tool review queue becomes a bottleneck and then a bypass.

**Control.** A policy on delegated authority instead of tool-by-tool approval.

**This lab.** Replace a per-tool review queue with a delegated-authority policy.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E3.2"))

Govern autonomy, not tools. A tool-approval process scales linearly with a list that grows weekly; an autonomy-tier policy does not.

In [ ]:
from cybercommons import planes

print(planes.describe_ladder())
print("\nPolicy expressed per rung rather than per tool:\n")
POLICY = {
 "L1":   "self-service. Register it. No further review.",
 "L2":   "register + named owner. Approval gate on every writer, enforced by policy.",
 "L2.5": "risk tier + blast-radius budget + drift monitoring + tested stop.",
 "L3":   "all of L2.5, plus held-out evaluation per release and board-level sign-off.",
}
for rung, rule in POLICY.items():
    print(f"  {rung:5s} {rule}")

Now test a request against it — which takes seconds and needs no tool committee.

In [ ]:
W = planes.Tool
request = planes.Manifest("new-triage-agent", [
    W("read_file"),
    W("post_comment", writes=True, scope="project"),
    W("close_ticket", writes=True, scope="project")], rung="L2")
print("requested rung:", request.rung)
problems = request.rung_check()
print("decision:", "approve at L2" if not problems else "refuse or re-tier —")
for p in problems:
    print("   ", p)
print(f"blast radius {request.blast_radius()['total']} "
      f"(budget for L2.5 in this example: 20)")

### Expect

The ladder and per-rung policy print, and the request is refused at L2 because both writers are ungated — with its blast radius shown against a budget.

### Your turn

Write your own per-rung policy in four lines. If L1 needs approval, nobody will register anything and your inventory dies.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E3.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*